<a href="https://colab.research.google.com/github/rafaelrdealmeida/fundamentos_2026_lantri01/blob/main/notebooks/encontro_2026_lantri_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introdução ao Pensamento computacional

## Ferramentas/Serviços utilizados

- Google Colab (notebook com codigo)
- Github (rede social de código)
- Git (programa de versionamento de código)


## Pilares
- Decomposição
- Abstração
- Identificação de padrões
- Lógica


## Noções Gerais

- Exemplos a partir de coleta de dados
  - Site MRE - Notas de imprensa
    - [x] Link do site: https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa
    - [x] Entender a estrutura da fonte de informação
    - [ ] Realizar a coleta
    - [ ] Inserir as informações em um banco de dados
    - [ ] Utilizar as informações (analise de dados)


### Padrão de paginação das notas de imprensa
  - https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=0
   - https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=30
   - Pagina final (10/02/2026): 6210



## Importação de bibliotecas/pogramas utilizados neste arquivo

In [1]:
!pip install tinydb

In [2]:
# programas/bibliotecas utilizados no script/codigo
import httpx # Responsável pelas requisições web
from bs4 import BeautifulSoup # Responsável por realizar o web scraping (coletar os dados)
from tinydb import TinyDB, Query

## Criação do banco json

In [3]:
def inserir_no_banco(dados, link_noticia):
  arquivo_banco_dados = "nota_mre.json"
  db = TinyDB(arquivo_banco_dados)


  # Evitar dados repetidos no banco
  Buscar = Query()
  verificar_link = db.contains(Buscar.link == link_noticia)

  if not verificar_link:
    print("Inserindo nova informação no banco")
    db.insert(dados)
  else:
    print("Link já existe no banco. Esta informação não será inserida novamente")

In [4]:
# Variável e tipos de dados (string, lista, numero)
paginas = ["https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=0"]

def acessa_pagina (link):
  print (f"Estamos na pagina:{link}")

  # Define headers para a requisição, simulando um navegador
  headers = {
      'User-Agent': "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36",
      'Accept-Language': 'en-US,en;q=0.9',
      'Accept-Encoding': 'gzip, deflate, br',
      'Connection': 'keep-alive',
  }

  timeout = httpx.Timeout(connect=20.0, read=30.0, write=20.0, pool=10.0)
  pag_web = httpx.get(link, headers=headers, timeout=timeout)
  bs = BeautifulSoup(pag_web, "html.parser")
  return bs

# loop for
# beautifulsoap >> find e find_all

for pagina in paginas:
  pagina_inteira = acessa_pagina(pagina)
  lista_noticias = pagina_inteira.find("div", attrs={"id": "content-core"}).find_all("article")
  for noticia in lista_noticias:
    # titulo
    try:
      titulo = noticia.find("h2", attrs={"class": "tileHeadline"}).text.strip()
      print(titulo)
    except:
        titulo = ""

    #link
    try:
      link_noticia = noticia.a["href"]
      print(link_noticia)
    except:
      link_noticia = ""
    # numero da nota - exemplo: NOTA À IMPRENSA Nº 72
    # numero da nota - exemplo: NOTA À IMPRENSA N° 590
    num_nota = noticia.find("span", attrs={"class": "subtitle"}).text.strip()
    # print(num_nota)
    # num_nota = noticia.find(attrs={"class": "subtitle"}).text.strip()
    num_nota = num_nota.replace("NOTA À IMPRENSA N°", "").replace("NOTA À IMPRENSA Nº", "").strip()
    print(num_nota)
    print("###")
    # data
    # horário
    data_hora = noticia.find_all("span",attrs={"class": "summary-view-icon"})
    data= data_hora[0].text.strip()
    hora = data_hora[1].text.strip()
    print(data)
    print(hora)
    conteudo = acessa_pagina (link_noticia)
    paragrafos = conteudo.find("div", attrs={"property":"rnews:articleBody"}).find_all("p")
    lista_paragrafos = []
    for paragrafo in paragrafos:
      lista_paragrafos.append(paragrafo.text.strip())
    print(lista_paragrafos)
    # função para inserir dados coletados no banco
    dados = {
        "titulo": titulo,
        "link": link_noticia,
        "data": data,
        "hora": hora,
        "num_nota": num_nota,
        "paragrafo": lista_paragrafos
    }
    inserir_no_banco(dados,link_noticia)






Estamos na pagina:https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=0
Redução dos valores cobrados de passaportes emitidos no exterior
https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/reducao-dos-valores-cobrados-de-passaportes-emitidos-no-exterior
154
###
04/05/2026
17h15
Estamos na pagina:https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/reducao-dos-valores-cobrados-de-passaportes-emitidos-no-exterior
['A partir de 1º de junho de 2026, os valores cobrados em moeda estrangeira para a emissão de passaportes brasileiros nas embaixadas e consulados do Brasil no exterior serão reduzidos pela metade, aproximando-os dos valores cobrados no Brasil.', 'A medida, formalizada pela Portaria MRE nº 664/2026, visa a contribuir para a manutenção da documentação brasileira em dia por parte de indivíduos e famílias binacionais no exterior, em especial de crianças nascidas fora do Brasil.', 'O passapo

# Transformar banco json e dataframe

- pre-analise - entendimento geral sobre o dataframe

In [5]:
import pandas as pd
import json

## Abrindo o rquivo json
with open("nota_mre.json") as f:
  raw = json.load(f)

df = pd.DataFrame.from_dict(raw["_default"], orient="index")

df


,titulo,link,data,hora,num_nota,paragrafo
1,Redução dos valores cobrados de passaportes em...,https://www.gov.br/mre/pt-br/canais_atendiment...,04/05/2026,17h15,154,"[A partir de 1º de junho de 2026, os valores c..."
2,IX Rodada Negociadora MERCOSUL-Canadá - Nota C...,https://www.gov.br/mre/pt-br/canais_atendiment...,01/05/2026,19h00,153,"[Realizou-se, entre os dias 27 e 30 de abril, ..."
3,Nota Conjunta Brasil-Espanha sobre sequestro e...,https://www.gov.br/mre/pt-br/canais_atendiment...,01/05/2026,12h00,152,"[Os governos do Brasil e da Espanha condenam, ..."
4,Declaração Conjunta sobre os Ataques Israelens...,https://www.gov.br/mre/pt-br/canais_atendiment...,30/04/2026,20h57,NOTA À IMPRENSA N. 151,"[., Declaração Conjunta dos Ministros das Rela..."
5,Abertura de mercado para o Brasil no Chile - N...,https://www.gov.br/mre/pt-br/canais_atendiment...,29/04/2026,17h49,150,[O governo brasileiro concluiu negociações que...
6,Promulgação do Acordo Provisório de Comércio e...,https://www.gov.br/mre/pt-br/canais_atendiment...,28/04/2026,19h20,149,[O Presidente Luiz Inácio Lula da Silva assino...
7,Mortes de brasileiros no Líbano em decorrência...,https://www.gov.br/mre/pt-br/canais_atendiment...,27/04/2026,19h04,148,"[O governo brasileiro tomou conhecimento, com ..."
8,Ataques terroristas no Mali,https://www.gov.br/mre/pt-br/canais_atendiment...,27/04/2026,13h38,147,[O Governo brasileiro manifesta grave preocupa...
9,Atentado na Colômbia,https://www.gov.br/mre/pt-br/canais_atendiment...,27/04/2026,08h45,146,[O governo brasileiro condena o ataque perpetr...
10,Apresentação da candidatura do Professor Georg...,https://www.gov.br/mre/pt-br/canais_atendiment...,24/04/2026,15h57,145,"[Realizou-se, em 23 de abril, coquetel de apre..."


In [6]:
# saber quantidade de linhas e colunas do dataframe
df.shape

(30, 6)

In [7]:
# saber colunas disponiveis
df.columns

Index(['titulo', 'link', 'data', 'hora', 'num_nota', 'paragrafo'], dtype='object')

In [8]:
# selecionar uma coluna em especifico
df["titulo"]

,titulo
1,Redução dos valores cobrados de passaportes em...
2,IX Rodada Negociadora MERCOSUL-Canadá - Nota C...
3,Nota Conjunta Brasil-Espanha sobre sequestro e...
4,Declaração Conjunta sobre os Ataques Israelens...
5,Abertura de mercado para o Brasil no Chile - N...
6,Promulgação do Acordo Provisório de Comércio e...
7,Mortes de brasileiros no Líbano em decorrência...
8,Ataques terroristas no Mali
9,Atentado na Colômbia
10,Apresentação da candidatura do Professor Georg...


In [9]:

# delimitar colunas do dataframe
df_delimitado = df[["titulo", "data"]]
df_delimitado

,titulo,data
1,Redução dos valores cobrados de passaportes em...,04/05/2026
2,IX Rodada Negociadora MERCOSUL-Canadá - Nota C...,01/05/2026
3,Nota Conjunta Brasil-Espanha sobre sequestro e...,01/05/2026
4,Declaração Conjunta sobre os Ataques Israelens...,30/04/2026
5,Abertura de mercado para o Brasil no Chile - N...,29/04/2026
6,Promulgação do Acordo Provisório de Comércio e...,28/04/2026
7,Mortes de brasileiros no Líbano em decorrência...,27/04/2026
8,Ataques terroristas no Mali,27/04/2026
9,Atentado na Colômbia,27/04/2026
10,Apresentação da candidatura do Professor Georg...,24/04/2026


In [10]:

# primeiras (head), ultimas (tail) e linhas aleatórias (sample)
df.head(10)

,titulo,link,data,hora,num_nota,paragrafo
1,Redução dos valores cobrados de passaportes em...,https://www.gov.br/mre/pt-br/canais_atendiment...,04/05/2026,17h15,154,"[A partir de 1º de junho de 2026, os valores c..."
2,IX Rodada Negociadora MERCOSUL-Canadá - Nota C...,https://www.gov.br/mre/pt-br/canais_atendiment...,01/05/2026,19h00,153,"[Realizou-se, entre os dias 27 e 30 de abril, ..."
3,Nota Conjunta Brasil-Espanha sobre sequestro e...,https://www.gov.br/mre/pt-br/canais_atendiment...,01/05/2026,12h00,152,"[Os governos do Brasil e da Espanha condenam, ..."
4,Declaração Conjunta sobre os Ataques Israelens...,https://www.gov.br/mre/pt-br/canais_atendiment...,30/04/2026,20h57,NOTA À IMPRENSA N. 151,"[., Declaração Conjunta dos Ministros das Rela..."
5,Abertura de mercado para o Brasil no Chile - N...,https://www.gov.br/mre/pt-br/canais_atendiment...,29/04/2026,17h49,150,[O governo brasileiro concluiu negociações que...
6,Promulgação do Acordo Provisório de Comércio e...,https://www.gov.br/mre/pt-br/canais_atendiment...,28/04/2026,19h20,149,[O Presidente Luiz Inácio Lula da Silva assino...
7,Mortes de brasileiros no Líbano em decorrência...,https://www.gov.br/mre/pt-br/canais_atendiment...,27/04/2026,19h04,148,"[O governo brasileiro tomou conhecimento, com ..."
8,Ataques terroristas no Mali,https://www.gov.br/mre/pt-br/canais_atendiment...,27/04/2026,13h38,147,[O Governo brasileiro manifesta grave preocupa...
9,Atentado na Colômbia,https://www.gov.br/mre/pt-br/canais_atendiment...,27/04/2026,08h45,146,[O governo brasileiro condena o ataque perpetr...
10,Apresentação da candidatura do Professor Georg...,https://www.gov.br/mre/pt-br/canais_atendiment...,24/04/2026,15h57,145,"[Realizou-se, em 23 de abril, coquetel de apre..."


In [11]:
df.describe(include="all")

,titulo,link,data,hora,num_nota,paragrafo
count,30,30,30,30,30,30
unique,30,30,16,28,30,30
top,Redução dos valores cobrados de passaportes em...,https://www.gov.br/mre/pt-br/canais_atendiment...,17/04/2026,19h20,154,"[A partir de 1º de junho de 2026, os valores c..."
freq,1,1,4,2,1,1


In [12]:
df.isnull().sum()

,0
titulo,0
link,0
data,0
hora,0
num_nota,0
paragrafo,0


In [13]:
# verificar linhas duplicadas
df.duplicated().sum()

TypeError: unhashable type: 'list'

### Tratamento de Dados e Visualização

Agora que entendemos a estrutura, vamos preparar os dados para extrair insights visuais. Um passo comum é converter colunas de texto em formatos que o computador entenda como 'tempo' (datetime) e criar gráficos.

In [ ]:
# 1. Converter a coluna de data para o formato datetime do Pandas
df['data_dt'] = pd.to_datetime(df['data'], dayfirst=True)

# 2. Contar quantas notícias temos por dia
noticias_por_dia = df['data_dt'].value_counts().sort_index()

display(noticias_por_dia)

#### Visualizando o Volume de Publicações

Vamos usar a biblioteca `matplotlib` (que já vem no ambiente) para criar um gráfico de barras simples.

In [ ]:
import matplotlib.pyplot as plt

# Criando o gráfico
plt.figure(figsize=(10, 5))
noticias_por_dia.plot(kind='bar', color='skyblue')

# Adicionando títulos e rótulos
plt.title('Quantidade de Notas à Imprensa por Data')
plt.xlabel('Data da Publicação')
plt.ylabel('Número de Notas')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

#### Analisando Palavras-Chave nos Títulos

Uma técnica simples de análise de texto para iniciantes é verificar a frequência de certas palavras-chave (como nomes de países).

In [ ]:
paises = ['Alemanha', 'Líbano', 'Vietnã', 'Togo', 'Haiti']
frequencia = {}

for pais in paises:
    # Conta em quantos títulos a palavra aparece
    frequencia[pais] = df['titulo'].str.contains(pais, case=False).sum()

# Converter para série para facilitar a plotagem
ser_freq = pd.Series(frequencia)

plt.figure(figsize=(8, 4))
ser_freq.sort_values(ascending=False).plot(kind='barh', color='lightgreen')
plt.title('Menções de Países nos Títulos das Notas')
plt.xlabel('Número de Menções')
plt.show()